In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://jobs.cvshealth.com/")
links

['search-results',
 'https://jobs.cvshealth.com/us/en/home',
 'https://jobs.cvshealth.com/us/en/home',
 'https://jobs.cvshealth.com/us/en/ftw-jobs',
 'javascript:void(0)',
 'https://jobs.cvshealth.com/us/en/careerareas',
 'https://jobs.cvshealth.com/us/en/life',
 'https://jobs.cvshealth.com/us/en/who-we-are',
 'javascript:void(0)',
 'https://jobs.cvshealth.com/us/en/jointalentcommunity',
 'javascript:void(0)',
 'javascript:void(0)',
 'https://jobs.cvshealth.com/us/en/hiring-process',
 'https://jobs.cvshealth.com/us/en/events',
 'javascript:void(0)',
 'https://jobs.cvshealth.com/us/en/blog',
 'javascript:void(0)',
 'javascript:void(0)',
 'https://jobs.cvshealth.com/us/en/search-results',
 'https://jobs.cvshealth.com/us/en/careerareas',
 'https://jobs.cvshealth.com/us/en/home',
 'https://cdn.phenompeople.com/CareerConnectResources/CVSCHLUS/documents/2025-CVS-EEO-Statement_wet-signature-1757513868342.pdf',
 'https://cdn.phenompeople.com/CareerConnectResources/CVSCHLUS/documents/2025-CVS-E

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://jobs.cvshealth.com/us/en/who-we-are"},
        {"type": "careers page", "url": "https://jobs.cvshealth.com/us/en/careerareas"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://jobs.cvshealth.com/us/en/home"))


Here is the list of links on the website https://jobs.cvshealth.com/us/en/home -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

search-results
https://jobs.cvshealth.com/us/en/home
https://jobs.cvshealth.com/us/en/home
https://jobs.cvshealth.com/us/en/ftw-jobs
javascript:void(0)
https://jobs.cvshealth.com/us/en/careerareas
https://jobs.cvshealth.com/us/en/life
https://jobs.cvshealth.com/us/en/who-we-are
javascript:void(0)
https://jobs.cvshealth.com/us/en/jointalentcommunity
javascript:void(0)
javascript:void(0)
https://jobs.cvshealth.com/us/en/hiring-process
https://jobs.cvshealth.com/us/en/events
javascript:void(0)
https://jobs.cvshealth.com/us/en/blog
javascript:void(0)
javascript:void(0)
https://jobs.cvshealth.com/us/en/search-results
https://jobs.cvshealth.com/us/en/careerareas
https://jobs.cvshealth.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [ ]:
select_relevant_links("https://jobs.cvshealth.com/us/en/home")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media page', 'url': 'https://twitter.com/edwarddonner'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [8]:
select_relevant_links("https://jobs.cvshealth.com/us/en/home")

Selecting relevant links for https://jobs.cvshealth.com/us/en/home by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'about page',
   'url': 'https://jobs.cvshealth.com/us/en/who-we-are'},
  {'type': 'life page', 'url': 'https://jobs.cvshealth.com/us/en/life'},
  {'type': 'careers page', 'url': 'https://jobs.cvshealth.com/us/en/ftw-jobs'},
  {'type': 'careers page',
   'url': 'https://jobs.cvshealth.com/us/en/careerareas'},
  {'type': 'talent community',
   'url': 'https://jobs.cvshealth.com/us/en/jointalentcommunity'},
  {'type': 'hiring process',
   'url': 'https://jobs.cvshealth.com/us/en/hiring-process'},
  {'type': 'events page', 'url': 'https://jobs.cvshealth.com/us/en/events'},
  {'type': 'blog page', 'url': 'https://jobs.cvshealth.com/us/en/blog'},
  {'type': 'social media', 'url': 'https://www.facebook.com/CVSCareers/'},
  {'type': 'social media',
   'url': 'https://www.linkedin.com/showcase/cvs-health-careers'},
  {'type': 'social media', 'url': 'https://www.instagram.com/cvscareers'},
  {'type': 'social media',
   'url': 'https://www.youtube.com/user/CVSPharmacyVideos'}

In [32]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relavent_links = select_relevant_links(url)
    result = f"Landing Page:\n\n{contents}\n\n Relavent Links:\n"
    for link in relavent_links["links"]:
        result += f"\nLink : {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [33]:
print(fetch_page_and_all_relevant_links("https://jobs.cvshealth.com/us/en/home"))

Selecting relevant links for https://jobs.cvshealth.com/us/en/home by calling gpt-5-nano
Found 7 relevant links
Landing Page:

CVS Health Careers

Search results
Warehouse, Fulfillment, and Transportation
United States
Drive incredible results to make healthier happen.
Search Warehouse, Fulfillment, and Transportation Jobs
We’re making healthier happen and welcome all heart-minded talent.
We're making healthier happen and
welcome all purpose-driven talent
We’re building a world of health around every consumer
, delivering superior care and value with heart. We care for people where, when and how they choose in a way that is uniquely more connected, more convenient and more compassionate. We care at our core, and we welcome all purpose-driven talent to serve our communities.
Live your purpose. Grow your career.
Make your impact.
We’re building a world of health around every individual — shaping a more connected, convenient and compassionate health experience. At CVS Health®, you’ll work

In [34]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [35]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [36]:
get_brochure_user_prompt("CVS Health", "https://jobs.cvshealth.com/us/en/home")

Selecting relevant links for https://jobs.cvshealth.com/us/en/home by calling gpt-5-nano
Found 7 relevant links


"\n    You are looking at a company called: CVS Health\n    Here are the contents of its landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code blocks.\n\n\n    Landing Page:\n\nCVS Health Careers\n\nSearch results\nWarehouse, Fulfillment, and Transportation\nUnited States\nDrive incredible results to make healthier happen.\nSearch Warehouse, Fulfillment, and Transportation Jobs\nWe’re making healthier happen and welcome all heart-minded talent.\nWe're making healthier happen and\nwelcome all purpose-driven talent\nWe’re building a world of health around every consumer\n, delivering superior care and value with heart. We care for people where, when and how they choose in a way that is uniquely more connected, more convenient and more compassionate. We care at our core, and we welcome all purpose-driven talent to serve our communities.\nLive your purpose. Grow your career.\nMake your impact.\nWe’re building a wo

In [39]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = [
            {"role":"system", "content": brochure_system_prompt},
            {"role":"user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    display(Markdown(response.choices[0].message.content))

In [40]:
create_brochure("CVS Health", "https://jobs.cvshealth.com/us/en/home")

Selecting relevant links for https://jobs.cvshealth.com/us/en/home by calling gpt-5-nano
Found 5 relevant links


# CVS Health - Building a World of Health Around You

---

## About CVS Health

CVS Health is a leading healthcare innovator dedicated to delivering high-quality, affordable healthcare to millions of Americans. At our core, we are driven by a purpose to make health and care better—more connected, convenient, and compassionate. Across every part of our business, we work together to lower healthcare costs, improve access, and provide superior care experiences that empower individuals and families to live healthier lives.

Our unique portfolio includes:
- **Aetna®**: Serving over 26 million medical members, providing simple, personal, and meaningful health plan experiences.
- **Coram®**: Specialists in home infusion and nutrition therapy with highly trained compassionate clinicians.
- **CVS Caremark®**: The leading pharmacy benefit manager in the U.S., committed to lowering medication costs and expanding affordable access to prescriptions.

---

## Our Customers

We proudly serve millions of customers nationwide by delivering healthcare where and when they need it—whether in one of our many retail locations, via specialty infusion services at home, or through supportive health plans. Our presence in local communities creates trusted relationships where individuals and families feel cared for and supported on their health journeys.

---

## Company Culture & Values

At CVS Health, we believe that making health and care better starts with people who bring heart and purpose to work every day. With a workforce of over 300,000 colleagues, we foster a culture of mutual respect, mindfulness, and innovation. Our teams are passionate about overcoming challenges, transforming healthcare, and ensuring safety and quality in everything we do.

Culture highlights include:
- Commitment to inclusivity and diversity
- Supportive, purpose-driven work environment
- Focus on innovation to simplify healthcare experiences
- Strong emphasis on compassionate care

---

## Careers at CVS Health

Join a team that is passionate about advancing the future of healthcare. CVS Health offers diverse career paths across areas such as pharmacy, retail, technology, warehouse fulfillment, transportation, and health services. Whether you are in-store helping customers directly, working behind the scenes to manage supply chains, or innovating through technology, your work will make a real impact.

Why work with us?
- Meaningful, purpose-driven work improving lives every day
- Opportunities for career growth and development
- Comprehensive and inclusive benefits supporting employees and their families
- A dynamic environment where your ideas and talents are valued

Explore career opportunities and join us in making a healthier future happen.

---

## Join Us

CVS Health is more than a company—it’s a commitment to better health for every person, every family, and every community. Bring your heart, grow your career, and make your impact with us.

**Discover your opportunity at:** [CVS Health Careers](https://jobs.cvshealth.com)

---

*CVS Health: Simplifying health care one person at a time.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## Who We Are  
**Hugging Face** is the premier AI community dedicated to building the future of machine learning. We are a vibrant platform where machine learning enthusiasts, researchers, and organizations come together to collaborate on models, datasets, and applications. Our mission is to empower the next generation of machine learning engineers, scientists, and users by fostering open, ethical, and innovative AI development.

---

## Our Platform  
Hugging Face is the **home of machine learning** — a collaboration hub that hosts over **2 million models**, **500k+ datasets**, and **1 million+ applications** spanning all modalities: text, image, video, audio, and even 3D. Our open-source stack accelerates your AI development and enables seamless sharing and experimentation.

- **Collaboration & Hosting:** Unlimited public models, datasets, and application hosting.
- **Explore & Build:** Share your work to build your machine learning portfolio and profile.
- **Spaces & Buckets:** Run AI applications easily with scalable compute options including free access with ZeroGPU and private storage.
- **Trending Models & Applications:** Stay updated with the latest, popular machine learning models and innovative AI apps.

---

## Enterprise Solutions  
Scale your organization with Hugging Face’s **Team** and **Enterprise** plans, designed for professional and large-scale AI projects.

- **Pricing:** Team plans start at $20/user/month with flexible Enterprise contracts available.
- **Enterprise-grade Security:** Single Sign-On (SSO), advanced security policies, audit logs, and granular access control.
- **Centralized Management:** Token management, resource groups, private dataset viewer, and dedicated analytics dashboards.
- **Enhanced Performance:** Additional compute options, 5x ZeroGPU quota boost, and private storage for your organization.
- **Priority Support:** Expert assistance to maximize your AI platform usage.
- Trusted by leading AI organizations for secure, scalable collaboration.

---

## Company Culture & Community  
At Hugging Face, we foster an **open, collaborative, and ethical AI community**. Our platform revolves around knowledge-sharing and pushing the boundaries of what AI can achieve together in an inclusive environment. We support creativity, transparency, and learning — helping individuals and teams build impactful AI solutions.

---

## Careers  
Join a world-class team passionate about advancing the machine learning frontier! Hugging Face offers exciting roles across research, engineering, product, and community engagement.

- **Current Openings:** Visit the Hugging Face Careers page to explore opportunities.
- Build your career in a mission-driven company shaping the AI landscape.
- Work alongside experts and contribute to open source projects with global impact.

---

## Why Choose Hugging Face?  
- Access to the world’s largest curated collection of machine learning models and datasets.
- Platform designed for both individuals and organizations with advanced collaboration features.
- Flexible solutions from open community usage to enterprise-grade offerings.
- A diverse, supportive culture driving the ethical use and advancement of AI technology.

---

## Connect With Us  
- Explore models and datasets: https://huggingface.co/models  
- Learn about enterprise plans: https://huggingface.co/enterprise  
- Join our community and contribute: https://huggingface.co/spaces  
- Check career opportunities: https://huggingface.co/careers  

---

### Hugging Face  
*The AI community building the future.*

Colors:  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Grey: #6B7280

Logo icons and brand assets are available for partners and contributors.

---

**Discover, create, and innovate with Hugging Face — where machine learning lives and grows.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")